In [29]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
import seaborn as sns
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

import os

# ✅ Java 11 path
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/jdk-11.jdk/Contents/Home"

# ✅ Use your current Python environment for PySpark
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

In [30]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Elhub Production to Cassandra") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.cassandra.connection.port", "9042") \
    .config("spark.cassandra.auth.username", "cassandra") \
    .config("spark.cassandra.auth.password", "cassandra") \
    .config("spark.cassandra.output.consistency.level", "LOCAL_QUORUM") \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .getOrCreate()

print("✅ SparkSession successfully connected to Cassandra on localhost:9042")

✅ SparkSession successfully connected to Cassandra on localhost:9042


25/11/10 11:20:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
# %% [code]
import requests
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# --- API Settings ---
BASE_URL = "https://api.elhub.no/energy-data/v0/price-areas"
DATASET = "PRODUCTION_PER_GROUP_MBA_HOUR"
START_DATE = "2021-01-01"
END_DATE = "2021-12-31"

# --- Step 1: Get all available price areas dynamically ---
areas_response = requests.get(BASE_URL)
if areas_response.status_code == 200:
    areas = areas_response.json().get("priceAreas", ["NO1", "NO2", "NO3", "NO4", "NO5"])
else:
    print("⚠️ Could not fetch price areas from API. Using defaults (NO1-NO5).")
    areas = ["NO1", "NO2", "NO3", "NO4", "NO5"]

print(f"📍 Price areas to fetch: {areas}")


# --- Step 2: Function to fetch data for one area and time window ---
def fetch_data(area, start_date, end_date):
    api_url = f"{BASE_URL}/{area}"
    params = {
        "dataset": DATASET,
        "startDate": start_date,
        "endDate": end_date
    }
    response = requests.get(api_url, params=params)

    if response.status_code == 200:
        try:
            data = response.json()
            records = data["data"][0]["attributes"]["productionPerGroupMbaHour"]
            if records:
                df = pd.DataFrame(records)
                df["priceArea"] = area
                return df
            else:
                print(f"⚠️ No data returned for {area}: {start_date} → {end_date}")
                return pd.DataFrame()
        except (KeyError, IndexError):
            print(f"⚠️ Unexpected JSON structure for {area}: {start_date} → {end_date}")
            return pd.DataFrame()
    else:
        print(f"⚠️ Failed for {area}: {start_date} → {end_date} ({response.status_code})")
        return pd.DataFrame()


# --- Step 3: Loop over all months and all areas ---
all_data = []
start_dt = datetime.fromisoformat(START_DATE + "T00:00:00+02:00")
end_dt = datetime.fromisoformat(END_DATE + "T23:59:59+02:00")

for area in areas:
    current_start = start_dt
    while current_start < end_dt:
        current_end = current_start + relativedelta(months=1)
        if current_end > end_dt:
            current_end = end_dt

        start_str = current_start.isoformat()
        end_str = current_end.isoformat()

        print(f"Fetching {area}: {start_str} → {end_str}")
        df = fetch_data(area, start_str, end_str)
        if not df.empty:
            all_data.append(df)

        current_start = current_end


# --- Step 4: Combine and save all data ---
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv("elhub_production_2021_all_areas.csv", index=False)
    print(f"✅ Data for all price areas saved to elhub_production_2021_all_areas.csv")
else:
    print("⚠️ No data collected.")

# %% [markdown]
# ## 2️⃣ Insert into Cassandra using Spark
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DoubleType

schema = StructType([
    StructField("endTime", StringType(), True),
    StructField("lastUpdatedTime", StringType(), True),
    StructField("priceArea", StringType(), True),
    StructField("productionGroup", StringType(), True),
    StructField("quantityKwh", DoubleType(), True),
    StructField("startTime", StringType(), True),
])

# --- Convert Pandas DF to Spark DF ---
spark_df = spark.createDataFrame(final_df, schema=schema)

# --- Rename columns to match Cassandra primary key casing ---
spark_df = spark_df \
    .withColumnRenamed("endTime", "endtime") \
    .withColumnRenamed("lastUpdatedTime", "lastupdatedtime") \
    .withColumnRenamed("priceArea", "pricearea") \
    .withColumnRenamed("productionGroup", "productiongroup") \
    .withColumnRenamed("quantityKwh", "quantitykwh") \
    .withColumnRenamed("startTime", "starttime") \

# I ran this the first time, to make sure my code sent the data to cassandra. I also store a csv file locally on my computer
"""
# --- Write to Cassandra ---
spark_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(keyspace="elhub", table="production_2021") \
    .save()

"""